In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps (UTC)
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

logdf = pd.read_csv("log.csv")
logdf['timestamp'] = pd.to_datetime(logdf['timestamp'], unit='s', utc=True)

error_logs = pd.read_csv("error_logs.csv")
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) metric: unique cmdb_id list
metric_unique_cmdb = metric['cmdb_id'].dropna().unique().tolist()

# 2) metric: top 20 (cmdb_id, kpi_name) pairs by count with global P95 and P90
metric_grp = metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    count='count',
    global_P95_value=lambda x: float(x.quantile(0.95)) if len(x)>0 else None,
    global_P90_value=lambda x: float(x.quantile(0.90)) if len(x)>0 else None
).reset_index()
metric_top20_df = metric_grp.sort_values('count', ascending=False).head(20).reset_index(drop=True)
# JSON-serializable list of dicts
metric_top20 = metric_top20_df.to_dict(orient='records')

# 3) trace: top 20 (cmdb_id, trace_name) pairs by count with global P95
trace_grp = trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    count='count',
    global_P95_value=lambda x: float(x.quantile(0.95)) if len(x)>0 else None
).reset_index()
trace_top20_df = trace_grp.sort_values('count', ascending=False).head(20).reset_index(drop=True)
trace_top20 = trace_top20_df.to_dict(orient='records')

# 4) log: top 20 (cmdb_id, log_name) pairs by count with global P95
log_grp = logdf.groupby(['cmdb_id', 'log_name'])['value'].agg(
    count='count',
    global_P95_value=lambda x: float(x.quantile(0.95)) if len(x)>0 else None
).reset_index()
log_top20_df = log_grp.sort_values('count', ascending=False).head(20).reset_index(drop=True)
log_top20 = log_top20_df.to_dict(orient='records')

# 5) error_logs: unique cmdb_id values and top 20 messages with counts
error_unique_cmdb = error_logs['cmdb_id'].dropna().unique().tolist()
err_msg_counts = error_logs.groupby('message').size().reset_index(name='count')
err_top20_df = err_msg_counts.sort_values('count', ascending=False).head(20).reset_index(drop=True)
err_top20 = err_top20_df.to_dict(orient='records')

# Present compact JSON-serializable results (variables displayed)
metric_unique_cmdb, metric_top20, trace_top20, log_top20, error_unique_cmdb, err_top20

```
Out[1]:
```
import json

# Use existing variables from the prior execution environment:
# metric_unique_cmdb, metric_top20, trace_top20, log_top20, error_unique_cmdb, err_top20

n_services = len(metric_unique_cmdb) if 'metric_unique_cmdb' in globals() else 0

def top_examples(pairs, n=4, keys=None):
    examples = []
    for i, p in enumerate(pairs[:n]):
        if keys is None:
            keys = list(p.keys())
        # build a concise representation
        cmdb = p.get('cmdb_id', p.get('cmdb', ''))
        name = p.get('kpi_name') or p.get('trace_name') or p.get('log_name') or p.get('message', '')
        cnt = p.get('count', '')
        p95 = p.get('global_P95_value', '')
        examples.append(f"{cmdb}/{name} (count={cnt}, P95={p95})")
    return examples

metric_examples = top_examples(metric_top20, 4) if 'metric_top20' in globals() else []
trace_examples = top_examples(trace_top20, 4) if 'trace_top20' in globals() else []
log_examples = top_examples(log_top20, 4) if 'log_top20' in globals() else []

# Check error logs presence
error_msgs_present = False
if 'err_top20' in globals() and len(err_top20) > 0:
    error_msgs_present = True

summary = (
    f"Summary:\n"
    f"- Found {n_services} unique cmdb_id services in metric data.\n"
    f"- Top metric (cmdb_id/kpi) examples: {', '.join(metric_examples)}.\n"
    f"- Top trace examples: {', '.join(trace_examples)}. Notably, traces to ts-auth-service show high P95 latency (e.g. duration_p95 ≈ 1.92s), which stands out.\n"
    f"- Top log examples: {', '.join(log_examples)}. Logs show high row_counts for services like ts-auth-service and ts-order-service; error_count P95 is 0 for many services.\n"
    f"- Error logs: {'no error log messages found' if not error_msgs_present else 'error messages present; see counts'}.\n\n"
    f"Interpretation (straightforward): Tracing data highlights latency toward ts-auth-service (P95 ~1.9s) as a prominent anomaly to investigate first. Metric pairs also show elevated socket/diskio/memory values for some services (e.g., ts-voucher-service socket P95=4.0, ts-travel-service memory P95 large), so after investigating auth latency, check socket and memory/disk I/O on the listed services."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(['ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'], [{'cmdb_id': 'ts-voucher-service', 'kpi_name': 'socket', 'count': 25, 'global_P95_value': 4.0, 'global_P90_value': 4.0}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'cpu', 'count': 25, 'global_P95_value': 0.8298051890960151, 'global_P90_value': 0.8051796003029857}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'diskio', 'count': 25, 'global_P95_value': 61683.01055653906, 'global_P90_value': 60419.80875370434}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'error', 'count': 25, 'global_P95_value': 0.0, 'global_P90_value': 0.0}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'latency-50', 'count': 25, 'global_P95_value': 0.01795423528620304, 'global_P90_value': 0.0179061653620686}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'latency-90', 'count': 25, 'global_P95_value': 0.0243176235151654, 'global_P90_value': 0.02423109765172352}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'mem', 'count': 25, 'global_P95_value': 233168213.3333333, 'global_P90_value': 232622025.38666663}, {'cmdb_id': 'ts-travel2-mongo', 'kpi_name': 'mem', 'count': 25, 'global_P95_value': 67200262.3654054, 'global_P90_value': 67193869.65333334}, {'cmdb_id': 'ts-travel2-mongo', 'kpi_name': 'socket', 'count': 25, 'global_P95_value': 9.0, 'global_P90_value': 9.0}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'cpu', 'count': 25, 'global_P95_value': 5.173705054750224, 'global_P90_value': 5.059655624111136}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'diskio', 'count': 25, 'global_P95_value': 59659.7628714832, 'global_P90_value': 58732.87576043941}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'latency-50', 'count': 25, 'global_P95_value': 0.01951558231716098, 'global_P90_value': 0.01951014464426646}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'latency-90', 'count': 25, 'global_P95_value': 0.3734799948263077, 'global_P90_value': 0.372853245499459}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'mem', 'count': 25, 'global_P95_value': 277919274.9895496, 'global_P90_value': 277913094.82666665}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'socket', 'count': 25, 'global_P95_value': 21.33, 'global_P90_value': 21.116666666666667}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'diskio', 'count': 25, 'global_P95_value': 60690.00847140135, 'global_P90_value': 58240.86175627053}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-50', 'count': 25, 'global_P95_value': 0.018926575783617262, 'global_P90_value': 0.01883165192174186}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'count': 25, 'global_P95_value': 0.27949862586263946, 'global_P90_value': 0.26732350002259686}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'mem', 'count': 25, 'global_P95_value': 267759547.73333332, 'global_P90_value': 267757772.79999998}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'socket', 'count': 25, 'global_P95_value': 16.776666666666667, 'global_P90_value': 16.70945945945946}], [{'cmdb_id': 'ts-user-service', 'trace_name': 'trace.to_ts-user-service.row_count', 'count': 25, 'global_P95_value': 20.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-basic-info-service.duration_mean', 'count': 25, 'global_P95_value': 0.01332204052287574}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-basic-info-service.duration_p95', 'count': 25, 'global_P95_value': 0.02299891999999997}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-basic-info-service.error_rate', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-basic-info-service.row_count', 'count': 25, 'global_P95_value': 20.799999999999997}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-travel-service.duration_mean', 'count': 25, 'global_P95_value': 0.17041112}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-travel-service.duration_p95', 'count': 25, 'global_P95_value': 0.19912909999999995}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-travel-service.error_rate', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-admin-travel-service.row_count', 'count': 25, 'global_P95_value': 10.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-assurance-service.duration_mean', 'count': 25, 'global_P95_value': 0.006037999999999998}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-assurance-service.duration_p95', 'count': 25, 'global_P95_value': 0.010381359999999968}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-assurance-service.error_rate', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-assurance-service.row_count', 'count': 25, 'global_P95_value': 10.799999999999997}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-auth-service.duration_mean', 'count': 25, 'global_P95_value': 1.3790442373333331}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-auth-service.duration_p95', 'count': 25, 'global_P95_value': 1.9208695799999995}, {'cmdb_id': 'root', 'trace_name': 'trace.to_ts-auth-service.error_rate', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.duration_mean', 'count': 25, 'global_P95_value': 0.00760818468713562}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.duration_p95', 'count': 25, 'global_P95_value': 0.0127284999999999}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.error_rate', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-order-other-service.row_count', 'count': 25, 'global_P95_value': 83.0}], [{'cmdb_id': 'ts-admin-basic-info-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-admin-basic-info-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 31.599999999999994}, {'cmdb_id': 'ts-admin-travel-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-admin-travel-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 30.0}, {'cmdb_id': 'ts-assurance-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 16.0}, {'cmdb_id': 'ts-assurance-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-basic-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 2705.7999999999997}, {'cmdb_id': 'ts-basic-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 339.2}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-contacts-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 64.79999999999998}, {'cmdb_id': 'ts-contacts-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-consign-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-consign-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 68.59999999999998}, {'cmdb_id': 'ts-preserve-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 210.19999999999993}, {'cmdb_id': 'ts-preserve-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 466.0}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}, {'cmdb_id': 'ts-order-other-service', 'log_name': 'log.row_count', 'count': 25, 'global_P95_value': 498.79999999999995}, {'cmdb_id': 'ts-order-other-service', 'log_name': 'log.error_count', 'count': 25, 'global_P95_value': 0.0}], [], [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing 'metric' if available; otherwise load it
if 'metric' not in globals():
    metric = pd.read_csv("metric.csv")
    metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# 1) Compute global P95 for each (cmdb_id, kpi_name) for the candidate services (using full series)
candidate_metrics_thresholds_df = (
    metric[metric['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)
# Ensure float dtype
candidate_metrics_thresholds_df['global_P95'] = candidate_metrics_thresholds_df['global_P95'].astype(float)

# 2) Filter metric rows to the incident window (UTC inclusive)
start = pd.Timestamp('2024-02-24 04:11:36', tz='UTC')
end = pd.Timestamp('2024-02-24 04:41:36', tz='UTC')
filtered_metrics_window = metric[
    metric['cmdb_id'].isin(candidates) &
    (metric['timestamp'] >= start) &
    (metric['timestamp'] <= end)
].copy()

# 3) For each component-KPI, compute required summary fields
# Merge thresholds into the filtered window so we can compare values against global_P95
merged = filtered_metrics_window.merge(candidate_metrics_thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# Prepare aggregation per (cmdb_id, kpi_name)
def compute_summary(group):
    cmdb = group.name[0]
    kpi = group.name[1]
    global_p95 = float(group['global_P95'].iloc[0]) if not pd.isna(group['global_P95'].iloc[0]) else float('nan')
    points_in_window = int(len(group))
    # mask for anomalies: value >= global_p95 (if global_p95 is NaN, no anomaly)
    if np.isnan(global_p95):
        anomaly_mask = np.array([False]*len(group))
    else:
        anomaly_mask = group['value'].to_numpy() >= global_p95
    anomaly_count = int(anomaly_mask.sum())
    if anomaly_count > 0:
        earliest_ts = pd.to_datetime(group.loc[anomaly_mask, 'timestamp'].min())
        earliest_anomaly_timestamp = earliest_ts.isoformat()
    else:
        earliest_anomaly_timestamp = None
    max_value_in_window = float(group['value'].max()) if points_in_window > 0 else None
    if np.isnan(global_p95) or global_p95 == 0 or max_value_in_window is None:
        max_breach_ratio = None
    else:
        max_breach_ratio = float(max_value_in_window / global_p95)
    return pd.Series({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_P95': global_p95,
        'points_in_window': points_in_window,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_breach_ratio': max_breach_ratio
    })

if not merged.empty:
    summary_df = merged.groupby(['cmdb_id', 'kpi_name'], observed=True).apply(compute_summary).reset_index(drop=True)
else:
    # No rows in window at all
    summary_df = pd.DataFrame(columns=['cmdb_id','kpi_name','global_P95','points_in_window','anomaly_count',
                                       'earliest_anomaly_timestamp','max_value_in_window','max_breach_ratio'])

# 3b) Exclude component-KPIs that have points_in_window == 0
summary_df = summary_df[summary_df['points_in_window'] > 0].copy()

# 4) Sort by anomaly_count (desc) then max_breach_ratio (desc), limit to top 20
summary_df['max_breach_ratio_sort'] = summary_df['max_breach_ratio'].fillna(-1)  # for sorting None -> lowest
summary_df = summary_df.sort_values(by=['anomaly_count', 'max_breach_ratio_sort'], ascending=[False, False])
summary_df = summary_df.drop(columns=['max_breach_ratio_sort']).head(20).reset_index(drop=True)

# Prepare JSON-serializable outputs
# candidate_metrics_thresholds as list of dicts (for candidate components only)
candidate_metrics_thresholds = candidate_metrics_thresholds_df.to_dict(orient='records')

# summary as list of dicts, ensure native python types
def row_to_dict(r):
    return {
        'cmdb_id': str(r['cmdb_id']),
        'kpi_name': str(r['kpi_name']),
        'global_P95': (None if (r['global_P95'] is None or pd.isna(r['global_P95'])) else float(r['global_P95'])),
        'points_in_window': int(r['points_in_window']),
        'anomaly_count': int(r['anomaly_count']),
        'earliest_anomaly_timestamp': (r['earliest_anomaly_timestamp']),
        'max_value_in_window': (None if r['max_value_in_window'] is None or pd.isna(r['max_value_in_window']) else float(r['max_value_in_window'])),
        'max_breach_ratio': (None if r['max_breach_ratio'] is None or (isinstance(r['max_breach_ratio'], float) and np.isnan(r['max_breach_ratio'])) else float(r['max_breach_ratio']))
    }

summary_list = [row_to_dict(r) for _, r in summary_df.iterrows()]
candidate_metrics_thresholds_list = [
    {'cmdb_id': str(r['cmdb_id']), 'kpi_name': str(r['kpi_name']), 'global_P95': None if pd.isna(r['global_P95']) else float(r['global_P95'])}
    for r in candidate_metrics_thresholds_df.to_dict(orient='records')
]

# Store intermediates in kernel variables for later steps (as requested)
candidate_metrics_thresholds_df, filtered_metrics_window, summary_list, candidate_metrics_thresholds_list

```
Out[2]:
```
summary_lines = []

if 'summary_list' not in globals() or not summary_list:
    summary = "No anomalies detected for the selected candidate services in the given incident window."
else:
    top_n = min(5, len(summary_list))
    summary_lines.append(f"Analyzed candidate services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.")
    summary_lines.append(f"Filtered window: 2024-02-24T04:11:36Z → 2024-02-24T04:41:36Z. Total filtered metric rows for candidates: {len(filtered_metrics_window) if 'filtered_metrics_window' in globals() else 'unknown'}.")
    summary_lines.append("Top anomaly findings (sorted by anomaly_count then breach ratio):")
    for i in range(top_n):
        r = summary_list[i]
        breach = r['max_breach_ratio']
        breach_str = f"{breach:.3f}x" if breach is not None else "N/A"
        summary_lines.append(
            f"{i+1}. {r['cmdb_id']} / {r['kpi_name']}: anomaly_count={r['anomaly_count']}, "
            f"points_in_window={r['points_in_window']}, earliest_anomaly={r['earliest_anomaly_timestamp']}, "
            f"max_value={r['max_value_in_window']}, global_P95={r['global_P95']}, breach={breach_str}"
        )
    # Key interpretation and recommendation
    top = summary_list[0]
    if top['cmdb_id'] == 'ts-auth-service' and top['kpi_name'] == 'diskio':
        recommendation = (
            "Interpretation: The largest relative breach is a disk I/O spike on ts-auth-service "
            f"(max breach ≈ {top['max_breach_ratio']:.2f}x at {top['earliest_anomaly_timestamp']}). "
            "Other notable anomalies (smaller breaches ~1.03–1.36x) are CPU spikes on ts-travel-service and ts-order-service, "
            "and diskio increases on ts-route-service and ts-train-service.\n"
            "Recommendation: Investigate ts-auth-service disk I/O immediately (check disk saturation, I/O wait, or background I/O jobs), "
            "then validate CPU usage on ts-travel-service and ts-order-service and disk I/O on route/train services."
        )
    else:
        recommendation = (
            "Interpretation: Multiple small-to-moderate breaches detected across candidate services. "
            "Prioritize the component-KPI with the largest breach ratio above for investigation."
        )
    summary_lines.append(recommendation)
    # also include thresholds note
    n_thresholds = len(candidate_metrics_thresholds_list) if 'candidate_metrics_thresholds_list' in globals() else 0
    summary_lines.append(f"Computed global P95 thresholds for {n_thresholds} component-KPI pairs (used as anomaly thresholds).")

summary = "\n".join(summary_lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_P95
0     ts-auth-service         cpu  3.913645e+01
1     ts-auth-service      diskio  9.398501e+05
2     ts-auth-service  latency-50  1.620923e+00
3     ts-auth-service  latency-90  2.324185e+00
4     ts-auth-service         mem  2.563117e+08
..                ...         ...           ...
30  ts-travel-service  latency-50  1.892658e-02
31  ts-travel-service  latency-90  2.794986e-01
32  ts-travel-service         mem  2.677595e+08
33  ts-travel-service      socket  1.677667e+01
34  ts-travel-service    workload  1.050426e+01

[35 rows x 3 columns],                      timestamp            cmdb_id    kpi_name         value
43   2024-02-24 04:14:00+00:00    ts-auth-service         cpu  1.405021e+01
44   2024-02-24 04:14:00+00:00    ts-auth-service      diskio  4.897328e+04
45   2024-02-24 04:14:00+00:00    ts-auth-service  latency-50  2.704808e-01
46   2024-02-24 04:14:00+00:00    ts-auth-service  latency-90  7.637644e-01
47   2024-02-24 04:14:00+00:00    ts-auth-service         mem  2.339758e+08
...                        ...                ...         ...           ...
8928 2024-02-24 04:38:00+00:00  ts-travel-service  latency-50  1.872765e-02
8929 2024-02-24 04:38:00+00:00  ts-travel-service  latency-90  2.805038e-01
8930 2024-02-24 04:38:00+00:00  ts-travel-service         mem  2.677555e+08
8931 2024-02-24 04:38:00+00:00  ts-travel-service      socket  1.664865e+01
8932 2024-02-24 04:38:00+00:00  ts-travel-service    workload  1.124849e+01

[875 rows x 4 columns], [{'cmdb_id': 'ts-auth-service', 'kpi_name': 'diskio', 'global_P95': 939850.090857092, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 4101693.753758533, 'max_breach_ratio': 4.36419998642338}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'cpu', 'global_P95': 4.190007415693624, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 5.694697912551682, 'max_breach_ratio': 1.3591140414745466}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'cpu', 'global_P95': 4.123833528518637, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 5.603021479721567, 'max_breach_ratio': 1.3586924498706143}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'diskio', 'global_P95': 65782.56685225481, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 81114.41223292296, 'max_breach_ratio': 1.233068518215516}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'diskio', 'global_P95': 58065.77413353031, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:34:00+00:00', 'max_value_in_window': 65752.52444305443, 'max_breach_ratio': 1.1323800538996927}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'cpu', 'global_P95': 3.643134067311549, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 3.944916173980711, 'max_breach_ratio': 1.0828358498735848}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'workload', 'global_P95': 10.504259999999997, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 11.248486486486486, 'max_breach_ratio': 1.0708499681544907}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'diskio', 'global_P95': 60690.00847140135, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:19:00+00:00', 'max_value_in_window': 64516.60167482934, 'max_breach_ratio': 1.0630514527812462}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'diskio', 'global_P95': 59017.08708112196, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 62526.63456898455, 'max_breach_ratio': 1.0594666335029133}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-90', 'global_P95': 0.05797307724541007, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:19:00+00:00', 'max_value_in_window': 0.0614043967736876, 'max_breach_ratio': 1.0591881558012206}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'socket', 'global_P95': 16.776666666666667, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 17.541666666666668, 'max_breach_ratio': 1.0455990462944567}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'socket', 'global_P95': 12.959999999999999, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 13.5, 'max_breach_ratio': 1.0416666666666667}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'workload', 'global_P95': 7.111939999999998, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 7.354972972972973, 'max_breach_ratio': 1.0341725285889611}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'global_P95': 0.27949862586263946, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:22:00+00:00', 'max_value_in_window': 0.2884430924538051, 'max_breach_ratio': 1.032001826712241}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'workload', 'global_P95': 35.38047666666667, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 36.36003333333333, 'max_breach_ratio': 1.0276863614895708}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'workload', 'global_P95': 2.845753333333333, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 2.914483333333333, 'max_breach_ratio': 1.0241517770338493}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-50', 'global_P95': 1.620923494237475, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:28:00+00:00', 'max_value_in_window': 1.6582197630940527, 'max_breach_ratio': 1.0230092715598047}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-50', 'global_P95': 0.00925131409194868, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:19:00+00:00', 'max_value_in_window': 0.0094565909144385, 'max_breach_ratio': 1.0221889366688426}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'workload', 'global_P95': 22.01095666666667, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 22.4744, 'max_breach_ratio': 1.0210551199728255}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'socket', 'global_P95': 31.873333333333328, 'points_in_window': 25, 'anomaly_count': 2, 'earliest_anomaly_timestamp': '2024-02-24T04:28:00+00:00', 'max_value_in_window': 32.51351351351352, 'max_breach_ratio': 1.0200851342871846}], [{'cmdb_id': 'ts-auth-service', 'kpi_name': 'cpu', 'global_P95': 39.136451882656814}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'diskio', 'global_P95': 939850.090857092}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-50', 'global_P95': 1.620923494237475}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'latency-90', 'global_P95': 2.3241846988474952}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'mem', 'global_P95': 256311664.64000002}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'socket', 'global_P95': 31.873333333333328}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'workload', 'global_P95': 2.845753333333333}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'cpu', 'global_P95': 4.123833528518637}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'diskio', 'global_P95': 59017.08708112196}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-50', 'global_P95': 0.00925131409194868}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'latency-90', 'global_P95': 0.05797307724541007}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'mem', 'global_P95': 262582749.86666664}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'socket', 'global_P95': 12.959999999999999}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'workload', 'global_P95': 7.111939999999998}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'cpu', 'global_P95': 5.564484312076589}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'diskio', 'global_P95': 65782.56685225481}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'latency-50', 'global_P95': 0.005089792643961}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'latency-90', 'global_P95': 0.00917139296943162}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'mem', 'global_P95': 257807141.54666668}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'socket', 'global_P95': 17.763333333333332}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'workload', 'global_P95': 35.38047666666667}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'cpu', 'global_P95': 3.643134067311549}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'diskio', 'global_P95': 58065.77413353031}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-50', 'global_P95': 0.0072226452482324}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'latency-90', 'global_P95': 0.00956261176333124}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'mem', 'global_P95': 254216159.89621618}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'socket', 'global_P95': 25.226666666666663}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'workload', 'global_P95': 22.01095666666667}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'cpu', 'global_P95': 4.190007415693624}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'diskio', 'global_P95': 60690.00847140135}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-50', 'global_P95': 0.018926575783617262}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'global_P95': 0.27949862586263946}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'mem', 'global_P95': 267759547.73333332}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'socket', 'global_P95': 16.776666666666667}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'workload', 'global_P95': 10.504259999999997}])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing 'trace' if available; otherwise load it
if 'trace' not in globals():
    trace = pd.read_csv("trace.csv")
    trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Helper: pattern substrings to identify traces targeting a candidate service
target_subs = [f"to_{c}" for c in candidates]  # e.g., "to_ts-auth-service"

# 1) Identify trace_name values that target any candidate service and compute global P95 (on full series)
# Filter trace rows that mention any target substring in trace_name
mask_target = trace['trace_name'].astype(str).apply(lambda s: any(sub in s for sub in target_subs))
trace_candidates_full = trace[mask_target].copy()

# Compute global P95 per trace_name (FULL series as required)
trace_candidate_thresholds_df = (
    trace_candidates_full
    .groupby('trace_name', observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)
trace_candidate_thresholds_df['global_P95'] = trace_candidate_thresholds_df['global_P95'].astype(float)

# Extract target_service for each trace_name (e.g., 'to_ts-auth-service')
def extract_target_service(trace_name):
    for sub in target_subs:
        if sub in trace_name:
            return sub  # return 'to_ts-auth-service' etc.
    return None

trace_candidate_thresholds_df['target_service'] = trace_candidate_thresholds_df['trace_name'].apply(extract_target_service)

# 2) Filter trace.csv to the incident window (UTC inclusive) and store in filtered_traces_window
# Reuse start/end if present, otherwise define
if 'start' not in globals() or 'end' not in globals():
    start = pd.Timestamp('2024-02-24 04:11:36', tz='UTC')
    end = pd.Timestamp('2024-02-24 04:41:36', tz='UTC')

filtered_traces_window = trace[
    mask_target &
    (trace['timestamp'] >= start) &
    (trace['timestamp'] <= end)
].copy()

# 3) For each candidate trace_name, compute summary metrics within the window
# Merge thresholds into filtered window to compare values against global_P95
merged = filtered_traces_window.merge(trace_candidate_thresholds_df[['trace_name','global_P95','target_service']],
                                      on='trace_name', how='left')

def summarize_trace_group(g):
    trace_name = g.name
    target_service = g['target_service'].iloc[0] if 'target_service' in g.columns else None
    global_p95 = float(g['global_P95'].iloc[0]) if not pd.isna(g['global_P95'].iloc[0]) else float('nan')
    points_in_window = int(len(g))
    if np.isnan(global_p95):
        anomaly_mask = np.array([False]*len(g))
    else:
        anomaly_mask = g['value'].to_numpy() >= global_p95
    anomaly_count = int(anomaly_mask.sum())
    earliest_anomaly_timestamp = (pd.to_datetime(g.loc[anomaly_mask, 'timestamp'].min()).isoformat()
                                  if anomaly_count>0 else None)
    max_value_in_window = float(g['value'].max()) if points_in_window>0 else None
    if np.isnan(global_p95) or global_p95 == 0 or max_value_in_window is None:
        max_breach_ratio = None
    else:
        max_breach_ratio = float(max_value_in_window / global_p95)
    return pd.Series({
        'trace_name': trace_name,
        'target_service': target_service,
        'global_P95': (None if np.isnan(global_p95) else global_p95),
        'points_in_window': points_in_window,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_breach_ratio': max_breach_ratio
    })

if not merged.empty:
    trace_summary_df = merged.groupby('trace_name', observed=True).apply(summarize_trace_group).reset_index(drop=True)
else:
    trace_summary_df = pd.DataFrame(columns=['trace_name','target_service','global_P95','points_in_window',
                                             'anomaly_count','earliest_anomaly_timestamp','max_value_in_window','max_breach_ratio'])

# Exclude entries with points_in_window == 0
trace_summary_df = trace_summary_df[trace_summary_df['points_in_window'] > 0].copy()

# 4) Sort by anomaly_count desc then max_breach_ratio desc, limit to top 20
trace_summary_df['max_breach_ratio_sort'] = trace_summary_df['max_breach_ratio'].fillna(-1)
trace_summary_df = trace_summary_df.sort_values(by=['anomaly_count','max_breach_ratio_sort'], ascending=[False, False])
trace_summary_df = trace_summary_df.drop(columns=['max_breach_ratio_sort']).head(20).reset_index(drop=True)

# Prepare JSON-serializable list of dicts, ensure timestamps are ISO UTC strings
def row_to_dict(r):
    return {
        'trace_name': str(r['trace_name']),
        'target_service': (None if r['target_service'] is None else str(r['target_service'])),
        'global_P95': (None if r['global_P95'] is None else float(r['global_P95'])),
        'points_in_window': int(r['points_in_window']),
        'anomaly_count': int(r['anomaly_count']),
        'earliest_anomaly_timestamp': (r['earliest_anomaly_timestamp']),
        'max_value_in_window': (None if r['max_value_in_window'] is None else float(r['max_value_in_window'])),
        'max_breach_ratio': (None if r['max_breach_ratio'] is None else float(r['max_breach_ratio']))
    }

trace_summary_list = [row_to_dict(r) for _, r in trace_summary_df.iterrows()]

# Also prepare thresholds list (trace_name, target_service, global_P95)
trace_candidate_thresholds_list = [
    {'trace_name': str(r['trace_name']), 'target_service': r['target_service'], 'global_P95': None if pd.isna(r['global_P95']) else float(r['global_P95'])}
    for r in trace_candidate_thresholds_df.to_dict(orient='records')
]

# Expose key variables for the kernel (compact outputs)
trace_candidate_thresholds_df, filtered_traces_window, trace_summary_list, trace_candidate_thresholds_list

```
Out[3]:
```
Summary (plain English):

- I analyzed traces that target the five candidate services within the window 2024-02-24T04:11:36Z → 2024-02-24T04:41:36Z. The filtered trace rows for these targets in the window: 2,296.

- Note about error_rate traces: several error_rate traces have a global P95 of 0 and all window values equal 0. Those were counted as "anomalies" by the >= P95 rule but are not meaningful (they indicate no errors, not an error spike). Treat error_rate entries with P95==0 as non-actionable.

- Actionable findings (ranked by anomaly_count then breach ratio):
  1. trace.to_ts-order-service.row_count — global P95 457.35; points_in_window 174; anomaly_count 9; earliest anomaly 2024-02-24T04:15:00Z; max=627.0; max breach ≈ 1.37×. This indicates an elevated row_count (load) to ts-order-service starting ~04:15.
  2. trace.to_ts-travel-service.duration_p95 — global P95 0.8330; points_in_window 150; anomaly_count 8; earliest anomaly 2024-02-24T04:14:00Z; max=1.1601; max breach ≈ 1.39×. This is a notable P95 latency spike toward ts-travel-service.
  3. trace.to_ts-travel-service.row_count — global P95 1512.4; points_in_window 150; anomaly_count 8; earliest anomaly 2024-02-24T04:15:00Z; max=2081.0; max breach ≈ 1.38×. Elevated traffic to ts-travel-service.
  4. trace.to_ts-order-service.duration_p95 — global P95 0.3586; points_in_window 174; anomaly_count 9; earliest anomaly 2024-02-24T04:26:00Z; max=0.4651; max breach ≈ 1.30×. P95 latency increase for ts-order-service (peaks later ~04:26).
  5. trace.to_ts-order-service.duration_mean — similar pattern to duration_p95 (anomaly_count 9; earliest 04:26).
  6. trace.to_ts-route-service.duration_p95 — global P95 0.005005; points_in_window 100; anomaly_count 5; earliest anomaly 2024-02-24T04:14:00Z; max=0.01147; max breach ≈ 2.29×. Large relative jump in route-service P95 latency (but small absolute latency).
  7. Additional groups (train-service duration_p95/mean and row_count, auth-service row_count/duration_p95/mean) show smaller breaches (≈1.02–1.16×) and fewer anomalies.

- Interpretation and recommendation:
  - The trace data indicates two main issues during the incident window: increased load (row_count) toward ts-order-service and ts-travel-service, and elevated P95 latency for ts-travel-service and, later in the window, ts-order-service.
  - The largest relative latency spike (by factor) is trace.to_ts-route-service.duration_p95 (≈2.29×) but its absolute latency remains small; evaluate if that impacts downstream services.
  - Prioritize investigation in this order:
    1. ts-travel-service — investigate P95 latency and unusual row_count/traffic (earliest anomalies ~04:14–04:15).
    2. ts-order-service — investigate increased row_count and rising P95 latency (row_count spike at ~04:15; latency spike around ~04:26).
    3. ts-route-service — check the P95 latency jump (earliest ~04:14), though absolute latency is small.
    4. ts-train-service and ts-auth-service — lower-severity anomalies observed; check if related to traffic spikes.
  - Also cross-check metric signals (CPU, disk I/O, sockets) for these services during the same window to find resource bottlenecks causing trace latency.

If you want, I can produce the compact JSON list of the top trace summaries again or focus next on correlated metric KPIs for the top two traced services (ts-travel-service and ts-order-service).

The original code execution output of IPython Kernel is also provided below for reference:

(                                  trace_name   global_P95        target_service
0     trace.to_ts-auth-service.duration_mean     1.329369    to_ts-auth-service
1      trace.to_ts-auth-service.duration_p95     1.898694    to_ts-auth-service
2        trace.to_ts-auth-service.error_rate     0.000000    to_ts-auth-service
3         trace.to_ts-auth-service.row_count   336.000000    to_ts-auth-service
4    trace.to_ts-order-service.duration_mean     0.337031   to_ts-order-service
5     trace.to_ts-order-service.duration_p95     0.358628   to_ts-order-service
6       trace.to_ts-order-service.error_rate     0.000000   to_ts-order-service
7        trace.to_ts-order-service.row_count   457.350000   to_ts-order-service
8    trace.to_ts-route-service.duration_mean     0.004138   to_ts-route-service
9     trace.to_ts-route-service.duration_p95     0.005005   to_ts-route-service
10      trace.to_ts-route-service.error_rate     0.000000   to_ts-route-service
11       trace.to_ts-route-service.row_count  2070.300000   to_ts-route-service
12   trace.to_ts-train-service.duration_mean     0.004847   to_ts-train-service
13    trace.to_ts-train-service.duration_p95     0.006553   to_ts-train-service
14      trace.to_ts-train-service.error_rate     0.000000   to_ts-train-service
15       trace.to_ts-train-service.row_count  2580.600000   to_ts-train-service
16  trace.to_ts-travel-service.duration_mean     0.594935  to_ts-travel-service
17   trace.to_ts-travel-service.duration_p95     0.833044  to_ts-travel-service
18     trace.to_ts-travel-service.error_rate     0.000000  to_ts-travel-service
19      trace.to_ts-travel-service.row_count  1512.400000  to_ts-travel-service,                       timestamp             cmdb_id                               trace_name       value
12    2024-02-24 04:14:00+00:00                root   trace.to_ts-auth-service.duration_mean    0.343784
13    2024-02-24 04:14:00+00:00                root    trace.to_ts-auth-service.duration_p95    0.608955
14    2024-02-24 04:14:00+00:00                root      trace.to_ts-auth-service.error_rate    0.000000
15    2024-02-24 04:14:00+00:00                root       trace.to_ts-auth-service.row_count   35.000000
36    2024-02-24 04:14:00+00:00                root  trace.to_ts-order-service.duration_mean    0.229266
...                         ...                 ...                                      ...         ...
18767 2024-02-24 04:38:00+00:00  ts-travel2-service      trace.to_ts-route-service.row_count  315.000000
18776 2024-02-24 04:38:00+00:00  ts-travel2-service  trace.to_ts-train-service.duration_mean    0.004555
18777 2024-02-24 04:38:00+00:00  ts-travel2-service   trace.to_ts-train-service.duration_p95    0.005976
18778 2024-02-24 04:38:00+00:00  ts-travel2-service     trace.to_ts-train-service.error_rate    0.000000
18779 2024-02-24 04:38:00+00:00  ts-travel2-service      trace.to_ts-train-service.row_count  152.000000

[2296 rows x 4 columns], [{'trace_name': 'trace.to_ts-order-service.error_rate', 'target_service': 'to_ts-order-service', 'global_P95': 0.0, 'points_in_window': 174, 'anomaly_count': 174, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.0, 'max_breach_ratio': nan}, {'trace_name': 'trace.to_ts-travel-service.error_rate', 'target_service': 'to_ts-travel-service', 'global_P95': 0.0, 'points_in_window': 150, 'anomaly_count': 150, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.0, 'max_breach_ratio': nan}, {'trace_name': 'trace.to_ts-route-service.error_rate', 'target_service': 'to_ts-route-service', 'global_P95': 0.0, 'points_in_window': 100, 'anomaly_count': 100, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.0, 'max_breach_ratio': nan}, {'trace_name': 'trace.to_ts-train-service.error_rate', 'target_service': 'to_ts-train-service', 'global_P95': 0.0, 'points_in_window': 100, 'anomaly_count': 100, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.0, 'max_breach_ratio': nan}, {'trace_name': 'trace.to_ts-auth-service.error_rate', 'target_service': 'to_ts-auth-service', 'global_P95': 0.0, 'points_in_window': 50, 'anomaly_count': 50, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.0, 'max_breach_ratio': nan}, {'trace_name': 'trace.to_ts-order-service.row_count', 'target_service': 'to_ts-order-service', 'global_P95': 457.35, 'points_in_window': 174, 'anomaly_count': 9, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 627.0, 'max_breach_ratio': 1.3709412922269595}, {'trace_name': 'trace.to_ts-order-service.duration_p95', 'target_service': 'to_ts-order-service', 'global_P95': 0.3586281099999999, 'points_in_window': 174, 'anomaly_count': 9, 'earliest_anomaly_timestamp': '2024-02-24T04:26:00+00:00', 'max_value_in_window': 0.4651391499999999, 'max_breach_ratio': 1.2969957932187748}, {'trace_name': 'trace.to_ts-order-service.duration_mean', 'target_service': 'to_ts-order-service', 'global_P95': 0.33703056666666664, 'points_in_window': 174, 'anomaly_count': 9, 'earliest_anomaly_timestamp': '2024-02-24T04:26:00+00:00', 'max_value_in_window': 0.3884486, 'max_breach_ratio': 1.152561928853733}, {'trace_name': 'trace.to_ts-travel-service.duration_p95', 'target_service': 'to_ts-travel-service', 'global_P95': 0.8330441324999998, 'points_in_window': 150, 'anomaly_count': 8, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 1.16010725, 'max_breach_ratio': 1.3926119934588224}, {'trace_name': 'trace.to_ts-travel-service.row_count', 'target_service': 'to_ts-travel-service', 'global_P95': 1512.3999999999999, 'points_in_window': 150, 'anomaly_count': 8, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 2081.0, 'max_breach_ratio': 1.3759587410737901}, {'trace_name': 'trace.to_ts-travel-service.duration_mean', 'target_service': 'to_ts-travel-service', 'global_P95': 0.5949346522826087, 'points_in_window': 150, 'anomaly_count': 8, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 0.69961615, 'max_breach_ratio': 1.1759546150417626}, {'trace_name': 'trace.to_ts-route-service.duration_p95', 'target_service': 'to_ts-route-service', 'global_P95': 0.0050049399999999, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.01147465, 'max_breach_ratio': 2.29266484713108}, {'trace_name': 'trace.to_ts-train-service.duration_p95', 'target_service': 'to_ts-train-service', 'global_P95': 0.0065533299999999, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:27:00+00:00', 'max_value_in_window': 0.0075086499999999, 'max_breach_ratio': 1.1457762694691118}, {'trace_name': 'trace.to_ts-route-service.duration_mean', 'target_service': 'to_ts-route-service', 'global_P95': 0.0041384933430656645, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:14:00+00:00', 'max_value_in_window': 0.004733925, 'max_breach_ratio': 1.143876432212226}, {'trace_name': 'trace.to_ts-train-service.row_count', 'target_service': 'to_ts-train-service', 'global_P95': 2580.6, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 2872.0, 'max_breach_ratio': 1.1129194760908316}, {'trace_name': 'trace.to_ts-route-service.row_count', 'target_service': 'to_ts-route-service', 'global_P95': 2070.3, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:15:00+00:00', 'max_value_in_window': 2262.0, 'max_breach_ratio': 1.0925952760469497}, {'trace_name': 'trace.to_ts-train-service.duration_mean', 'target_service': 'to_ts-train-service', 'global_P95': 0.0048469912037037, 'points_in_window': 100, 'anomaly_count': 5, 'earliest_anomaly_timestamp': '2024-02-24T04:32:00+00:00', 'max_value_in_window': 0.0050635915492957, 'max_breach_ratio': 1.0446875879259883}, {'trace_name': 'trace.to_ts-auth-service.row_count', 'target_service': 'to_ts-auth-service', 'global_P95': 336.0, 'points_in_window': 50, 'anomaly_count': 4, 'earliest_anomaly_timestamp': '2024-02-24T04:16:00+00:00', 'max_value_in_window': 344.0, 'max_breach_ratio': 1.0238095238095237}, {'trace_name': 'trace.to_ts-auth-service.duration_p95', 'target_service': 'to_ts-auth-service', 'global_P95': 1.89869397, 'points_in_window': 50, 'anomaly_count': 3, 'earliest_anomaly_timestamp': '2024-02-24T04:26:00+00:00', 'max_value_in_window': 2.202768, 'max_breach_ratio': 1.160149047084191}, {'trace_name': 'trace.to_ts-auth-service.duration_mean', 'target_service': 'to_ts-auth-service', 'global_P95': 1.3293693251255412, 'points_in_window': 50, 'anomaly_count': 3, 'earliest_anomaly_timestamp': '2024-02-24T04:27:00+00:00', 'max_value_in_window': 1.440476352112676, 'max_breach_ratio': 1.0835787503797278}], [{'trace_name': 'trace.to_ts-auth-service.duration_mean', 'target_service': 'to_ts-auth-service', 'global_P95': 1.3293693251255412}, {'trace_name': 'trace.to_ts-auth-service.duration_p95', 'target_service': 'to_ts-auth-service', 'global_P95': 1.89869397}, {'trace_name': 'trace.to_ts-auth-service.error_rate', 'target_service': 'to_ts-auth-service', 'global_P95': 0.0}, {'trace_name': 'trace.to_ts-auth-service.row_count', 'target_service': 'to_ts-auth-service', 'global_P95': 336.0}, {'trace_name': 'trace.to_ts-order-service.duration_mean', 'target_service': 'to_ts-order-service', 'global_P95': 0.33703056666666664}, {'trace_name': 'trace.to_ts-order-service.duration_p95', 'target_service': 'to_ts-order-service', 'global_P95': 0.3586281099999999}, {'trace_name': 'trace.to_ts-order-service.error_rate', 'target_service': 'to_ts-order-service', 'global_P95': 0.0}, {'trace_name': 'trace.to_ts-order-service.row_count', 'target_service': 'to_ts-order-service', 'global_P95': 457.35}, {'trace_name': 'trace.to_ts-route-service.duration_mean', 'target_service': 'to_ts-route-service', 'global_P95': 0.0041384933430656645}, {'trace_name': 'trace.to_ts-route-service.duration_p95', 'target_service': 'to_ts-route-service', 'global_P95': 0.0050049399999999}, {'trace_name': 'trace.to_ts-route-service.error_rate', 'target_service': 'to_ts-route-service', 'global_P95': 0.0}, {'trace_name': 'trace.to_ts-route-service.row_count', 'target_service': 'to_ts-route-service', 'global_P95': 2070.3}, {'trace_name': 'trace.to_ts-train-service.duration_mean', 'target_service': 'to_ts-train-service', 'global_P95': 0.0048469912037037}, {'trace_name': 'trace.to_ts-train-service.duration_p95', 'target_service': 'to_ts-train-service', 'global_P95': 0.0065533299999999}, {'trace_name': 'trace.to_ts-train-service.error_rate', 'target_service': 'to_ts-train-service', 'global_P95': 0.0}, {'trace_name': 'trace.to_ts-train-service.row_count', 'target_service': 'to_ts-train-service', 'global_P95': 2580.6}, {'trace_name': 'trace.to_ts-travel-service.duration_mean', 'target_service': 'to_ts-travel-service', 'global_P95': 0.5949346522826087}, {'trace_name': 'trace.to_ts-travel-service.duration_p95', 'target_service': 'to_ts-travel-service', 'global_P95': 0.8330441324999998}, {'trace_name': 'trace.to_ts-travel-service.error_rate', 'target_service': 'to_ts-travel-service', 'global_P95': 0.0}, {'trace_name': 'trace.to_ts-travel-service.row_count', 'target_service': 'to_ts-travel-service', 'global_P95': 1512.3999999999999}])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if present
# logdf, error_logs, start, end were defined previously in the session
# If not, load them
if 'logdf' not in globals():
    logdf = pd.read_csv("log.csv")
    logdf['timestamp'] = pd.to_datetime(logdf['timestamp'], unit='s', utc=True)
if 'error_logs' not in globals():
    error_logs = pd.read_csv("error_logs.csv")
    error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)
if 'start' not in globals() or 'end' not in globals():
    start = pd.Timestamp('2024-02-24 04:11:36', tz='UTC')
    end = pd.Timestamp('2024-02-24 04:41:36', tz='UTC')

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-travel-service', 'ts-route-service', 'ts-train-service']

# 1) Compute global P95 for each (cmdb_id, log_name) using full series (before filtering)
log_thresholds = (
    logdf[logdf['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'log_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)
log_thresholds['global_P95'] = log_thresholds['global_P95'].astype(float)

# 2) Filter log.csv to the incident window and candidate services
filtered_logs_window = logdf[
    logdf['cmdb_id'].isin(candidates) &
    (logdf['timestamp'] >= start) &
    (logdf['timestamp'] <= end)
].copy()

# 3) For each (cmdb_id, log_name) with points in the window, compute summary metrics
# Merge thresholds into filtered logs
merged_logs = filtered_logs_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

def summarize_log_group(g):
    cmdb = g.name[0]
    log_name = g.name[1]
    global_p95 = float(g['global_P95'].iloc[0]) if not pd.isna(g['global_P95'].iloc[0]) else float('nan')
    points_in_window = int(len(g))
    if np.isnan(global_p95):
        anomaly_mask = np.array([False]*len(g))
    else:
        anomaly_mask = g['value'].to_numpy() >= global_p95
    anomaly_count = int(anomaly_mask.sum())
    earliest_anomaly_timestamp = (pd.to_datetime(g.loc[anomaly_mask, 'timestamp'].min()).isoformat()
                                  if anomaly_count>0 else None)
    max_value_in_window = float(g['value'].max()) if points_in_window>0 else None
    if np.isnan(global_p95) or global_p95 == 0 or max_value_in_window is None:
        max_breach_ratio = None
    else:
        max_breach_ratio = float(max_value_in_window / global_p95)
    return pd.Series({
        'cmdb_id': cmdb,
        'log_name': log_name,
        'global_P95': (None if np.isnan(global_p95) else global_p95),
        'points_in_window': points_in_window,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_breach_ratio': max_breach_ratio
    })

if not merged_logs.empty:
    log_summary_df = (
        merged_logs
        .groupby(['cmdb_id', 'log_name'], observed=True)
        .apply(summarize_log_group)
        .reset_index(drop=True)
    )
else:
    log_summary_df = pd.DataFrame(columns=['cmdb_id','log_name','global_P95','points_in_window','anomaly_count',
                                           'earliest_anomaly_timestamp','max_value_in_window','max_breach_ratio'])

# Exclude pairs with points_in_window == 0 (should already be excluded since grouped from merged_logs)
log_summary_df = log_summary_df[log_summary_df['points_in_window'] > 0].copy()

# Sort by anomaly_count desc then max_breach_ratio desc, limit to 20
log_summary_df['max_breach_sort'] = log_summary_df['max_breach_ratio'].fillna(-1)
log_summary_df = log_summary_df.sort_values(by=['anomaly_count', 'max_breach_sort'], ascending=[False, False])
log_summary_df = log_summary_df.drop(columns=['max_breach_sort']).head(20).reset_index(drop=True)

# 4) error_logs: filter to window and candidate services
error_filtered_window = error_logs[
    error_logs['cmdb_id'].isin(candidates) &
    (error_logs['timestamp'] >= start) &
    (error_logs['timestamp'] <= end)
].copy()

# (a) unique cmdb_id counts of error log rows (counts per service)
if not error_filtered_window.empty:
    error_counts_per_service = (
        error_filtered_window.groupby('cmdb_id', observed=True)
        .size()
        .reset_index(name='error_row_count')
    )
else:
    # produce empty DataFrame with the expected columns
    error_counts_per_service = pd.DataFrame({'cmdb_id': candidates, 'error_row_count': [0]*len(candidates)})
    # Ensure explicit zeros for services with no rows
    # But we'll adjust below to explicitly state none exist per-service

# (b) top 20 error messages with counts for these services in the window
if not error_filtered_window.empty:
    err_top20 = (
        error_filtered_window.groupby('message', observed=True)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .head(20)
        .reset_index(drop=True)
    )
else:
    err_top20 = pd.DataFrame(columns=['message','count'])

# For services with no error rows, ensure explicit statement: create a mapping
# Build a list of services with zero rows
error_counts_map = {row['cmdb_id']: int(row['error_row_count']) for _, row in error_counts_per_service.iterrows()}
# Ensure all candidate services are present
for svc in candidates:
    if svc not in error_counts_map:
        error_counts_map[svc] = 0

# Prepare compact JSON-serializable outputs:
# Convert log_summary_df to a list of dicts with proper types
def to_output_list(df):
    rows = []
    for _, r in df.iterrows():
        rows.append({
            'cmdb_id': str(r['cmdb_id']),
            'log_name': str(r['log_name']),
            'global_P95': (None if r['global_P95'] is None or pd.isna(r['global_P95']) else float(r['global_P95'])),
            'points_in_window': int(r['points_in_window']),
            'anomaly_count': int(r['anomaly_count']),
            'earliest_anomaly_timestamp': r['earliest_anomaly_timestamp'],
            'max_value_in_window': (None if r['max_value_in_window'] is None or pd.isna(r['max_value_in_window']) else float(r['max_value_in_window'])),
            'max_breach_ratio': (None if r['max_breach_ratio'] is None or (isinstance(r['max_breach_ratio'], float) and np.isnan(r['max_breach_ratio'])) else float(r['max_breach_ratio']))
        })
    return rows

log_summary_list = to_output_list(log_summary_df)

# Prepare log_thresholds list (compact) for these services
log_thresholds_list = [
    {'cmdb_id': str(r['cmdb_id']), 'log_name': str(r['log_name']), 'global_P95': None if pd.isna(r['global_P95']) else float(r['global_P95'])}
    for r in log_thresholds.to_dict(orient='records')
]

# Prepare error top messages list
err_top20_list = err_top20.to_dict(orient='records')

# For services with no error messages, we will explicitly list them with a note (in a small dict)
services_with_no_error_msgs = [svc for svc, cnt in error_counts_map.items() if cnt == 0]

# Expose variables (compact outputs). Per rules, avoid showing full raw frames; show thresholds (small), summary list (small), and error counts/top messages.
log_thresholds, filtered_logs_window.shape, pd.DataFrame(log_summary_list).head(20), error_counts_map, err_top20_list, services_with_no_error_msgs

```
Out[4]:
```
summary = (
    "Summary of log and error-log analysis for candidate services (window 2024-02-24T04:11:36Z → 2024-02-24T04:41:36Z):\n\n"
    "1) Global log thresholds (P95, computed from full series):\n"
    "   - log.row_count P95: ts-travel-service=2008.8, ts-order-service=466.0, ts-route-service=1079.6, ts-auth-service=339.2\n"
    "   - log.error_count P95 = 0 for all listed services (log.error_count baseline is zero).\n\n"
    "2) Windowed log anomalies (only pairs with data in the window):\n"
    "   - ts-travel-service / log.row_count: 25 points; anomaly_count=2; earliest anomaly=2024-02-24T04:15:00+00:00; "
    "max=2313.0 (≈1.151× above P95).\n"
    "   - ts-order-service / log.row_count: 25 points; anomaly_count=2; earliest anomaly=2024-02-24T04:15:00+00:00; "
    "max=528.0 (≈1.133× above P95).\n"
    "   - ts-route-service / log.row_count: 25 points; anomaly_count=2; earliest anomaly=2024-02-24T04:15:00+00:00; "
    "max=1131.0 (≈1.048× above P95).\n"
    "   - ts-auth-service / log.row_count: 25 points; anomaly_count=2; earliest anomaly=2024-02-24T04:17:00+00:00; "
    "max=344.0 (≈1.014× above P95).\n"
    "   - For log.error_count, window values are all zero; they were counted as '>= P95' because P95==0 but are non-actionable (no errors observed).\n\n"
    "3) Error logs (error_logs.csv) within the window for these services: none found. Each candidate service has 0 error-log rows in this window.\n\n"
    "Interpretation and next steps:\n"
    "- There were short-lived increases in log.row_count (traffic/load) to ts-travel-service and ts-order-service around 04:15Z, with smaller increases for ts-route and ts-auth. "
    "These row_count spikes correspond to the trace findings (elevated row_count and P95 latency for travel and order).\n"
    "- No actual error messages were present in error logs during the window; log.error_count remained zero.\n"
    "- Recommended action: investigate elevated traffic/throughput to ts-travel-service and ts-order-service at ~04:15Z and check related resource metrics (CPU, disk I/O, sockets) for those services to find bottlenecks causing the trace latencies."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  global_P95
0    ts-auth-service  log.error_count         0.0
1    ts-auth-service    log.row_count       339.2
2   ts-order-service  log.error_count         0.0
3   ts-order-service    log.row_count       466.0
4   ts-route-service  log.error_count         0.0
5   ts-route-service    log.row_count      1079.6
6  ts-travel-service  log.error_count         0.0
7  ts-travel-service    log.row_count      2008.8, (200, 4),              cmdb_id         log_name  global_P95  points_in_window  anomaly_count earliest_anomaly_timestamp  max_value_in_window  max_breach_ratio
0    ts-auth-service  log.error_count         0.0                25             25  2024-02-24T04:14:00+00:00                  0.0               NaN
1   ts-order-service  log.error_count         0.0                25             25  2024-02-24T04:14:00+00:00                  0.0               NaN
2   ts-route-service  log.error_count         0.0                25             25  2024-02-24T04:14:00+00:00                  0.0               NaN
3  ts-travel-service  log.error_count         0.0                25             25  2024-02-24T04:14:00+00:00                  0.0               NaN
4  ts-travel-service    log.row_count      2008.8                25              2  2024-02-24T04:15:00+00:00               2313.0          1.151434
5   ts-order-service    log.row_count       466.0                25              2  2024-02-24T04:15:00+00:00                528.0          1.133047
6   ts-route-service    log.row_count      1079.6                25              2  2024-02-24T04:15:00+00:00               1131.0          1.047610
7    ts-auth-service    log.row_count       339.2                25              2  2024-02-24T04:17:00+00:00                344.0          1.014151, {'ts-auth-service': 0, 'ts-order-service': 0, 'ts-travel-service': 0, 'ts-route-service': 0, 'ts-train-service': 0}, [], ['ts-auth-service', 'ts-order-service', 'ts-travel-service', 'ts-route-service', 'ts-train-service'])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import re

# Use existing filtered_traces_window if present; otherwise filter 'trace' to the incident window
if 'filtered_traces_window' in globals() and isinstance(filtered_traces_window, pd.DataFrame):
    traces_window = filtered_traces_window.copy()
else:
    if 'trace' not in globals():
        trace = pd.read_csv("trace.csv")
        trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
    if 'start' not in globals() or 'end' not in globals():
        start = pd.Timestamp('2024-02-24 04:11:36', tz='UTC')
        end = pd.Timestamp('2024-02-24 04:41:36', tz='UTC')
    traces_window = trace[(trace['timestamp'] >= start) & (trace['timestamp'] <= end)].copy()

# 1) Parse target service from trace_name (token after 'trace.to_' up to next dot)
traces_window['target_service'] = traces_window['trace_name'].astype(str).str.extract(r'trace\.to_([^\.]+)', expand=False)

# Normalize None for missing targets
traces_window['target_service'] = traces_window['target_service'].where(traces_window['target_service'].notna(), None)

# 2) Build caller->target edges where caller = cmdb_id and target = parsed target_service
edges = traces_window[traces_window['target_service'].notna()].copy()
edges['caller_cmdb'] = edges['cmdb_id'].astype(str)

# Candidate services
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# 3) Mark whether target_service is one of the candidate services
edges['is_candidate_target'] = edges['target_service'].apply(lambda t: True if (t in candidates) else False)

# 2 continued) Aggregate by (caller, target): count and earliest timestamp
agg = (
    edges
    .groupby(['caller_cmdb', 'target_service', 'is_candidate_target'], observed=True)
    .agg(count=('trace_name', 'count'), earliest_timestamp=('timestamp', 'min'))
    .reset_index()
)

# Convert earliest_timestamp to ISO UTC strings safely
agg['earliest_timestamp'] = agg['earliest_timestamp'].dt.tz_convert('UTC').apply(lambda x: x.isoformat() if pd.notna(x) else None)

# 4) Sort by count desc and return top 20 edges with required fields
edges_summary_df = agg.sort_values(by='count', ascending=False).head(20).reset_index(drop=True)
edges_summary_df = edges_summary_df[['caller_cmdb', 'target_service', 'is_candidate_target', 'count', 'earliest_timestamp']]

# Display compact JSON-serializable result
edges_summary_df.head(20)

```
Out[5]:
```
Summary (plain English):

- The top call edges in the incident window show very high-volume inbound traffic to the candidate services. The highest-count edges are:
  - root → ts-auth-service, root → ts-order-service, and root → ts-travel-service (each count=100, earliest 2024-02-24T04:14:00Z).
- Many internal services also call the candidates: e.g., ts-admin-travel-service, ts-food-service, ts-preserve-service, ts-seat-service, and ts-travel-service itself all call ts-travel-service; several services call ts-order-service as well.
- ts-travel-service and ts-order-service appear as frequent downstream targets from many callers (multiple distinct callers, earliest appearances ~04:14–04:15Z), indicating they are hot spots in high-volume call chains during the incident.
- ts-auth-service is also a high-volume target from root and others, but the strongest multi-caller concentration is on ts-travel-service (many different callers).
- Earliest activity for these edges is around 04:14Z, matching the earlier trace/log anomalies.
- Recommendation: prioritize investigating ts-travel-service and ts-order-service first (look at their trace P95 spikes, log.row_count spikes, and resource metrics like CPU/disk I/O), then validate effects on ts-auth-service.

The original code execution output of IPython Kernel is also provided below for reference:

caller_cmdb     target_service  is_candidate_target  count         earliest_timestamp
0                      root    ts-auth-service                 True    100  2024-02-24T04:14:00+00:00
1                      root   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
2                      root  ts-travel-service                 True    100  2024-02-24T04:14:00+00:00
3   ts-admin-travel-service  ts-travel-service                 True    100  2024-02-24T04:14:00+00:00
4           ts-auth-service    ts-auth-service                 True    100  2024-02-24T04:14:00+00:00
5          ts-basic-service   ts-route-service                 True    100  2024-02-24T04:14:00+00:00
6          ts-basic-service   ts-train-service                 True    100  2024-02-24T04:14:00+00:00
7           ts-food-service  ts-travel-service                 True    100  2024-02-24T04:14:00+00:00
8          ts-order-service   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
9       ts-preserve-service   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
10      ts-preserve-service  ts-travel-service                 True    100  2024-02-24T04:14:00+00:00
11          ts-seat-service   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
12         ts-route-service   ts-route-service                 True    100  2024-02-24T04:14:00+00:00
13          ts-seat-service  ts-travel-service                 True    100  2024-02-24T04:14:00+00:00
14      ts-security-service   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
15        ts-travel-service   ts-train-service                 True    100  2024-02-24T04:14:00+00:00
16         ts-train-service   ts-train-service                 True    100  2024-02-24T04:14:00+00:00
17        ts-travel-service   ts-order-service                 True    100  2024-02-24T04:14:00+00:00
18        ts-travel-service   ts-route-service                 True    100  2024-02-24T04:14:00+00:00
19       ts-travel2-service   ts-route-service                 True    100  2024-02-24T04:14:00+00:00```
```